# Traditional tabular-model benchmark

This notebook trains XGBoost and LightGBM on every dataset in `data/pre-processed`. It saves one metrics CSV per model and a held-out-test confusion matrix for every model/dataset pair.

## 1. Import libraries and configure paths

In [1]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    make_scorer,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET_COLUMN = "target"
DATASETS = ("diabetes", "heart", "hepatitis")
CV_FOLDS = 5

# Locate the repository whether Jupyter starts in the notebook directory or project root.
repository_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data" / "pre-processed").exists()),
    None,
)
if repository_root is None:
    raise FileNotFoundError("Could not find data/pre-processed.")

data_dir = repository_root / "data" / "pre-processed"
results_dir = repository_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

## 2. Load data and define evaluation helpers

A stratified 5-fold cross-validation search selects the best basic parameter combination using mean validation macro-F1. The result files report these cross-validation aggregates with a `train_` prefix, followed by held-out test metrics and separate training and prediction times.

In [2]:
def load_dataset(dataset_name: str):
    """Load one preprocessed train/test pair and separate features from target."""
    train_data = pd.read_csv(data_dir / f"{dataset_name}_train.csv")
    test_data = pd.read_csv(data_dir / f"{dataset_name}_test.csv")

    X_train = train_data.drop(columns=TARGET_COLUMN)
    y_train = train_data[TARGET_COLUMN].astype(int)
    X_test = test_data.drop(columns=TARGET_COLUMN)
    y_test = test_data[TARGET_COLUMN].astype(int)
    return X_train, y_train, X_test, y_test


def calculate_metrics(y_true: pd.Series, probabilities) -> dict:
    """Calculate binary-classification metrics using a 0.5 probability threshold."""
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "f1_macro": f1_score(y_true, predictions, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "auc": roc_auc_score(y_true, probabilities),
    }


def save_confusion_matrix(y_true: pd.Series, probabilities, model_name: str, dataset_name: str):
    """Save the test-set confusion matrix for one model/dataset pair."""
    predictions = (probabilities >= 0.5).astype(int)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        predictions,
        display_labels=["No bad event (0)", "Bad event (1)"],
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"{model_name} — {dataset_name.title()} test confusion matrix")
    fig.tight_layout()
    model_results_dir = results_dir / model_name
    model_results_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(model_results_dir / f"{model_name}_{dataset_name}_CM.png", dpi=150)
    plt.close(fig)


SCORING = {
    "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "auc": "roc_auc",
}

## 3. Tune models with cross-validation, evaluate, and save results

In [3]:
model_search_spaces = {
    "xgboost": {
        "estimator": XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", n_estimators=200,
            subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=1,
        ),
        # Four compact, sensible configurations.
        "param_grid": {"max_depth": [3, 4], "learning_rate": [0.05, 0.1]},
    },
    "lightgbm": {
        "estimator": LGBMClassifier(
            objective="binary", n_estimators=200, max_depth=4, subsample=0.8,
            colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=1, verbosity=-1,
        ),
        # Four compact, sensible configurations.
        "param_grid": {"num_leaves": [7, 15], "learning_rate": [0.05, 0.1]},
    },
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for model_name, search_space in model_search_spaces.items():
    model_results = []

    for dataset_name in DATASETS:
        X_train, y_train, X_test, y_test = load_dataset(dataset_name)
        search = GridSearchCV(
            estimator=search_space["estimator"], param_grid=search_space["param_grid"],
            scoring=SCORING, refit="f1_macro", cv=cv, n_jobs=-1,
        )

        start_time = perf_counter()
        search.fit(X_train, y_train)
        train_time_seconds = perf_counter() - start_time

        best_index = search.best_index_
        train_metrics = {
            metric: search.cv_results_[f"mean_test_{metric}"][best_index]
            for metric in SCORING
        }
        prediction_start_time = perf_counter()
        test_probabilities = search.best_estimator_.predict_proba(X_test)[:, 1]
        prediction_time_seconds = perf_counter() - prediction_start_time
        test_metrics = calculate_metrics(y_test, test_probabilities)

        row = {
            "dataset": dataset_name,
            "train_time_seconds": train_time_seconds,
            "prediction_time_seconds": prediction_time_seconds,
            "best_parameters": str(search.best_params_),
            **{f"train_{metric}": value for metric, value in train_metrics.items()},
            **{f"test_{metric}": value for metric, value in test_metrics.items()},
        }
        model_results.append(row)
        save_confusion_matrix(y_test, test_probabilities, model_name, dataset_name)

    results = pd.DataFrame(model_results)
    model_results_dir = results_dir / model_name
    model_results_dir.mkdir(parents=True, exist_ok=True)
    results.to_csv(model_results_dir / f"{model_name}_results.csv", index=False)
    print(f"Saved {model_name}_results.csv")
    display(results.round(4))

Saved xgboost_results.csv


,dataset,train_time_seconds,prediction_time_seconds,best_parameters,train_f1_macro,train_accuracy,train_precision,train_recall,train_auc,test_f1_macro,test_accuracy,test_precision,test_recall,test_auc
0,diabetes,3.0538,0.0041,"{'learning_rate': 0.1, 'max_depth': 4}",0.6160,0.9256,0.5243,0.1861,0.8347,0.6191,0.9317,0.6667,0.1724,0.8229
1,heart,0.3992,0.0019,"{'learning_rate': 0.1, 'max_depth': 4}",0.8124,0.8155,0.8190,0.7852,0.8879,0.8075,0.8103,0.8000,0.7692,0.8690
2,hepatitis,0.3699,0.0033,"{'learning_rate': 0.05, 'max_depth': 3}",0.7404,0.8297,0.6667,0.5467,0.8372,0.6900,0.8065,0.5000,0.5000,0.8800


Saved lightgbm_results.csv


,dataset,train_time_seconds,prediction_time_seconds,best_parameters,train_f1_macro,train_accuracy,train_precision,train_recall,train_auc,test_f1_macro,test_accuracy,test_precision,test_recall,test_auc
0,diabetes,0.6581,0.0066,"{'learning_rate': 0.1, 'num_leaves': 15}",0.6175,0.9269,0.5845,0.1859,0.8236,0.5988,0.9253,0.5000,0.1552,0.8385
1,heart,0.2212,0.0014,"{'learning_rate': 0.05, 'num_leaves': 15}",0.8039,0.8070,0.8164,0.7671,0.8937,0.8057,0.8103,0.8261,0.7308,0.8822
2,hepatitis,0.1908,0.0017,"{'learning_rate': 0.05, 'num_leaves': 7}",0.7322,0.8307,0.6838,0.5067,0.8569,0.7567,0.8387,0.5714,0.6667,0.7933
